In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from catboost import CatBoostRegressor
from collections import deque

In [3]:
CB_MODEL_NAME = "catboost_delta.cbm"
LOOKBACK = 5
FORWARD = 5
MIN_MOVEMENT = 10

model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\115\weights\best.pt')
cap = cv2.VideoCapture(0)
track_history = {}
dataset_rows = []

print("Начат сбор данных. Двигай маркер...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.5, verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            if track_id not in track_history:
                track_history[track_id] = deque(maxlen=LOOKBACK + FORWARD + 1)
            
            track = track_history[track_id]
            track.append((float(x), float(y)))

            for i in range(1, len(track)):
                cv2.line(frame, (int(track[i-1][0]), int(track[i-1][1])), 
                         (int(track[i][0]), int(track[i][1])), (0, 0, 255), 2)

            if len(track) >= (LOOKBACK + FORWARD):
                past_pts = list(track)[-(LOOKBACK + FORWARD):-FORWARD] 
                future_pt = track[-1] 
                
                dist = np.linalg.norm(np.array(past_pts[0]) - np.array(past_pts[-1]))
                
                if dist > MIN_MOVEMENT:

                    row = []
                    for i in range(1, len(past_pts)):
                        row.extend([past_pts[i][0] - past_pts[i-1][0], 
                                    past_pts[i][1] - past_pts[i-1][1]])
                    
                    target_dx = future_pt[0] - past_pts[-1][0]
                    target_dy = future_pt[1] - past_pts[-1][1]
                    
                    dataset_rows.append(row + [target_dx, target_dy])

    cv2.putText(frame, f"Collected: {len(dataset_rows)}", (10, 30), 1, 1, (0,255,0), 2)
    cv2.imshow("Data Collection Mode", frame)
    
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

if len(dataset_rows) > 100:
    print(f"\nСбор завершен. Обучаю модель на {len(dataset_rows)} строках...")
    data = np.array(dataset_rows)
    X, Y = data[:, :-2], data[:, -2:]
    
    cb_model = CatBoostRegressor(
        iterations=2000, 
        loss_function='MultiRMSE', 
        depth=6, 
        learning_rate=0.05,
        verbose=100
    )
    
    cb_model.fit(X, Y)
    cb_model.save_model(CB_MODEL_NAME)
    print(f"Готово! Модель сохранена как {CB_MODEL_NAME}")
else:
    print("\nСлишком мало данных.")


Начат сбор данных. Двигай маркер...

Сбор завершен. Обучаю модель на 5080 строках...
0:	learn: 69.1055123	total: 128ms	remaining: 4m 15s
100:	learn: 45.8971240	total: 520ms	remaining: 9.78s
200:	learn: 42.4225731	total: 955ms	remaining: 8.55s
300:	learn: 40.1261913	total: 1.34s	remaining: 7.57s
400:	learn: 38.4327515	total: 1.74s	remaining: 6.93s
500:	learn: 37.0525892	total: 2.15s	remaining: 6.43s
600:	learn: 35.8094778	total: 2.63s	remaining: 6.12s
700:	learn: 34.6675699	total: 3.11s	remaining: 5.76s
800:	learn: 33.7106575	total: 3.6s	remaining: 5.39s
900:	learn: 32.8377720	total: 4.05s	remaining: 4.94s
1000:	learn: 32.0459435	total: 4.53s	remaining: 4.52s
1100:	learn: 31.2464913	total: 4.98s	remaining: 4.06s
1200:	learn: 30.5091798	total: 5.46s	remaining: 3.63s
1300:	learn: 29.7990551	total: 5.91s	remaining: 3.17s
1400:	learn: 29.1152601	total: 6.33s	remaining: 2.71s
1500:	learn: 28.5147753	total: 6.71s	remaining: 2.23s
1600:	learn: 27.9105227	total: 7.07s	remaining: 1.76s
1700:	lea